In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

/home/nesaulov/study/urfu/coursach/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/data.csv')
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [3]:
mass = '232,03806	12,011	51,9961	58,933194	95,95	183,84	26,9815385	47,867	92,90637	10,81	55,845	88,90584	91,224	180,94788	186,207	101,07	50,9415	140,116	138,90547	32,06	28,085	54,938044	24,305	30,973761998	178,49	107,8682	63,546	208,9804	207,2	192,22	72,63	69,723	58,6934'.replace(',', '.').split()
element = 'Th	C	Cr	Co	Mo	W	Al	Ti	Nb	B	Fe	Y	Zr	Ta	Re	Ru	V	Ce	La	S	Si	Mn	Mg	P	Hf	Ag	Cu	Bi	Pb	Ir	Ge	Ga	Ni'.split()
atom_mass = dict(zip(element, mass))

In [4]:
atom_mass['Ni']

'58.6934'

In [ ]:
for elem in df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list():
    df[elem] = df[elem] / float(atom_mass[elem])
df['sum'] = df[df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list()].sum(axis=1, skipna=True)
# df['PLM'] = df['T.K'] * (20 + np.log10(df['t.h'])) * 1e-5

cols = df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns
df.loc[:, cols] = df.loc[:, cols].div(df['sum'], axis=0)
df.loc[:, cols] = df.loc[:, cols].div(df['Ni'], axis=0)



df = df.fillna(0)

# df = df.drop(columns=['T.K', 't.h', 'Ni', 'sum'])
df = df.drop(columns=['Ni','sum'])
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,PLM
0,241.316495,1144.2600,4.5,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001652,0.000000,0.0,0.000000,0.139405
1,241.316495,1144.2600,113.4,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.002485,0.000000,0.0,0.000000,0.149239
2,241.316495,1144.2600,68.6,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001653,0.000000,0.0,0.000423,0.147465
3,241.316495,1144.2600,40.7,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001656,0.000000,0.0,0.001695,0.146154
4,241.316495,1144.2600,32.4,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001661,0.000000,0.0,0.004252,0.145925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.368295
3442,1103.161120,866.4833,25554.7,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.366118
3443,241.316495,1199.8170,183.2,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.262391
3444,241.316495,1199.8170,153.0,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.261469


In [ ]:
def compute_P_LM(T_K, tau_h):
    # P_LM = T_K * (20 + log10(tau_h)) * 1e-5
    return T_K * (20.0 + np.log10(tau_h)) * 1e-5

def y_from_sigma_sigma_mpa(sigma_mpa):
    # y = -log10(sigma)
    return -np.log10(sigma_mpa)

def sigma_from_y(y):
    # sigma = 10^(-y)
    return 10 ** (-y)

def relative_rmse_sigma(sigma_pred, sigma_true):
    # RMSE relative: sqrt(mean(((pred-true)/true)^2))
    return np.sqrt(np.mean(((sigma_pred - sigma_true) / sigma_true) ** 2))

In [7]:
class BRANN(nn.Module):
    def __init__(self, input_dim, hidden_dim=13, prior_mu=0.0, prior_sigma=0.1):
        super().__init__()
        self.fc1 = bnn.BayesLinear(
            in_features=input_dim, out_features=hidden_dim,
            prior_mu=prior_mu, prior_sigma=prior_sigma
        )
        self.act = nn.Sigmoid()  # ближе к "sigmoid" из статьи (можно заменить на Tanh)
        self.fc2 = bnn.BayesLinear(
            in_features=hidden_dim, out_features=1,
            prior_mu=prior_mu, prior_sigma=prior_sigma
        )

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

In [8]:
def train_brann_bootstrap(
    X, y, input_dim,
    ensemble_seed,
    hidden_dim=13,
    bootstrap_iters=10,       # "up to 10 iterations"
    epochs_per_iter=10,      # "up to 10 training epochs"
    val_ratio=0.25,          # 75/25 split
    lr=1e-2,
    kl_weight=0.01,         # см. в demo: kl_weight
    prior_sigma=0.1,
):
    """
    Возвращает: (best_model, best_val_mse) для одной сети ансамбля.
    Для близости к статье: внутри сети делаем bootstrap_iters итераций,
    каждую обучаем epochs_per_iter эпох и выбираем лучшую по val MSE.
    """
    rng = np.random.default_rng(ensemble_seed)
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).reshape(-1, 1)
    n = len(X)

    best_val_mse_global = float("inf")
    best_state = None

    for it in range(bootstrap_iters):
        # bootstrap-like: train 75% sample, val remaining 25%
        train_size = int(n * (1 - val_ratio))
        train_idx = rng.choice(n, size=train_size, replace=True)

        # val: use those not in train_idx unique; если оказалось пусто — берём случайно
        val_candidates = np.setdiff1d(np.arange(n), np.unique(train_idx))
        if len(val_candidates) == 0:
            val_idx = rng.choice(n, size=max(1, n - train_size), replace=False)
        else:
            val_idx = rng.choice(val_candidates, size=max(1, n - train_size), replace=False)

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        # init new model each bootstrap iteration
        model = BRANN(input_dim=input_dim, hidden_dim=hidden_dim, prior_sigma=prior_sigma).to(device)

        mse_loss = nn.MSELoss()
        kl_loss = bnn.BKLLoss(reduction="mean", last_layer_only=False)

        opt = optim.Adam(model.parameters(), lr=lr)

        X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
        X_val_t = torch.tensor(X_val, dtype=torch.float32, device=device)
        y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)

        best_val_mse_local = float("inf")
        best_state_local = None

        model.train()
        for epoch in range(epochs_per_iter):
            pred = model(X_train_t)
            mse = mse_loss(pred, y_train_t)
            kl = kl_loss(model)
            cost = mse + kl_weight * kl

            opt.zero_grad()
            cost.backward()
            opt.step()

            # val mse
            model.eval()
            with torch.no_grad():
                pred_val = model(X_val_t)
                val_mse = mse_loss(pred_val, y_val_t).item()

            model.train()
            if val_mse < best_val_mse_local:
                best_val_mse_local = val_mse
                best_state_local = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        # choose best among bootstrap iterations for this ensemble member
        if best_val_mse_local < best_val_mse_global:
            best_val_mse_global = best_val_mse_local
            best_state = best_state_local

    # load best state
    best_model = BRANN(input_dim=input_dim, hidden_dim=hidden_dim, prior_sigma=prior_sigma).to(device)
    best_model.load_state_dict(best_state)
    return best_model, best_val_mse_global


In [9]:
@torch.no_grad()
def ensemble_predict(models, X):
    X_t = torch.tensor(np.asarray(X, dtype=np.float32), device=device)
    preds_y = []
    for m in models:
        m.eval()
        yhat = m(X_t).detach().cpu().numpy().reshape(-1)  # y-space
        preds_y.append(yhat)
    preds_y = np.stack(preds_y, axis=0)  # (k, n)
    y_mean = preds_y.mean(axis=0)
    return y_mean

In [10]:
def run_BRANN_article_like(
    df: pd.DataFrame,
    composition_cols,   # list of N columns
    T_col,              # temperature column (Kelvin)
    tau_col,           # time column (hours)
    sigma_col,         # Sigma, MPa column
    hidden_dim=13,
    ensemble_size=7,
    bootstrap_iters=10,
    epochs_per_iter=10,
    lr=1e-2,
    kl_weight=0.01,
    prior_sigma=0.1,
    test_ratio=0.2,
    seed=42,
):
    # 1) build dataset and filter invalid rows
    T = df[T_col].to_numpy(dtype=float)
    tau = df[tau_col].to_numpy(dtype=float)
    sigma = df[sigma_col].to_numpy(dtype=float)

    mask = (T > 0) & (tau > 0) & (sigma > 0)  # log requirements
    df2 = df.loc[mask].copy()

    comp = df2[composition_cols].to_numpy(dtype=float)
    T2 = df2[T_col].to_numpy(dtype=float)
    tau2 = df2[tau_col].to_numpy(dtype=float)
    sigma2 = df2[sigma_col].to_numpy(dtype=float)

    # 2) compute P_LM and target y
    P_LM = compute_P_LM(T2, tau2).reshape(-1, 1)
    X = np.hstack([comp, P_LM])  # input_dim = 27 + 1 = 28

    y = y_from_sigma_sigma_mpa(sigma2).reshape(-1)  # target y

    input_dim = X.shape[1]

    # 3) honest split: train+val vs test (как внешний тест)
    X_trainval, X_test, y_trainval, y_test, sigma_trainval, sigma_test = train_test_split(
        X, y, sigma2, test_size=test_ratio, random_state=seed, shuffle=True
    )

    # scale
    scaler_X = StandardScaler()
    X_trainval_s = scaler_X.fit_transform(X_trainval)
    X_test_s = scaler_X.transform(X_test)
    
    scaler_y = StandardScaler()
    y_trainval_s = scaler_y.fit_transform(y_trainval.reshape(-1, 1))
    y_test_s = scaler_y.transform(y_test.reshape(-1, 1))

    print("sigma_test min/mean/median/max:",
      sigma_test.min(), sigma_test.mean(), np.median(sigma_test), sigma_test.max())


    # 4) train ensemble (each ensemble member trained on trainval)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device == "cuda":
        torch.cuda.manual_seed_all(seed)

    models = []
    for k in tqdm(range(ensemble_size), desc="Training BRANN ensemble (scaled)"):
        model, _ = train_brann_bootstrap(
            X_trainval_s, y_trainval_s,
            input_dim=input_dim,
            ensemble_seed=seed + 1000 + k,
            hidden_dim=hidden_dim,
            bootstrap_iters=bootstrap_iters,
            epochs_per_iter=epochs_per_iter,
            val_ratio=0.25,
            lr=lr,
            kl_weight=kl_weight,
            prior_sigma=prior_sigma,
        )
        models.append(model)

    # 5) predict on test: average in y then convert to sigma
    y_pred_test_s = ensemble_predict(models, X_test_s)        # scaled y
    y_pred_test = scaler_y.inverse_transform(y_pred_test_s.reshape(-1, 1)).reshape(-1)  # back to y

    sigma_pred_test = sigma_from_y(y_pred_test)  # MPa

    # 6) metrics in sigma space
    mae = np.mean(np.abs(sigma_pred_test - sigma_test))
    rmse = np.sqrt(np.mean((sigma_pred_test - sigma_test) ** 2))
    rmse_rel = relative_rmse_sigma(sigma_pred_test, sigma_test)

    return {
        "MAE_MPa": mae,
        "RMSE_MPa": rmse,
        "RMSE_relative": rmse_rel,
        "sigma_pred_test": sigma_pred_test,
        "sigma_test": sigma_test,
        "y_pred_test": y_pred_test,
        "y_test": y_test.reshape(-1),
        "scaler_X": scaler_X,
        "scaler_y": scaler_y,
        "models": models,
    }

In [11]:
res = run_BRANN_article_like(
    df=df,
    composition_cols=df.columns.to_list(),
    T_col="T.K",
    tau_col="t.h",
    sigma_col="Sigma, Mpa",
    hidden_dim=13,
    ensemble_size=7,
    bootstrap_iters=10,
    epochs_per_iter=10,
    lr=1e-2,
    kl_weight=0.01,
    prior_sigma=0.1,
    test_ratio=0.2,
    seed=42
)

print(res["MAE_MPa"], res["RMSE_MPa"], res["RMSE_relative"])

sigma_test min/mean/median/max: 10.0 334.583196735314 270.0 1279.0


Training BRANN ensemble (scaled): 100%|██████████| 7/7 [00:05<00:00,  1.25it/s]

154.51230785705232 206.39997664466253 1.3291839631199187


In [12]:
err_rel = (res["sigma_pred_test"] - res["sigma_test"]) / res["sigma_test"]   # в долях
abs_err = np.abs(res["sigma_pred_test"] - res["sigma_test"])
abs_rel_err = np.abs(err_rel)


In [13]:
idx_sorted = np.argsort(abs_rel_err)[::-1]
top = 20  # сколько худших точек показать

print("Worst relative errors:")
for i in idx_sorted[:top]:
    print(i,
          "sigma_true=", res["sigma_test"][i],
          "sigma_pred=", res["sigma_pred_test"][i],
          "abs_err=", abs_err[i],
          "rel_err=", err_rel[i])


Worst relative errors:
461 sigma_true= 10.0 sigma_pred= 219.82593 abs_err= 209.825927734375 rel_err= 20.9825927734375
653 sigma_true= 21.0 sigma_pred= 212.47913 abs_err= 191.4791259765625 rel_err= 9.118053617931547
503 sigma_true= 30.0 sigma_pred= 207.55716 abs_err= 177.55715942382812 rel_err= 5.918571980794271
678 sigma_true= 35.0 sigma_pred= 232.05519 abs_err= 197.05519104003906 rel_err= 5.630148315429688
284 sigma_true= 31.0 sigma_pred= 202.16113 abs_err= 171.1611328125 rel_err= 5.521326864919355
225 sigma_true= 40.0 sigma_pred= 246.0823 abs_err= 206.08230590820312 rel_err= 5.152057647705078
34 sigma_true= 39.0 sigma_pred= 234.77583 abs_err= 195.7758331298828 rel_err= 5.019893157176482
510 sigma_true= 38.0 sigma_pred= 220.4735 abs_err= 182.47349548339844 rel_err= 4.80193409166838
357 sigma_true= 40.0 sigma_pred= 226.60672 abs_err= 186.60671997070312 rel_err= 4.665167999267578
273 sigma_true= 35.4 sigma_pred= 196.99266 abs_err= 161.59266052246093 rel_err= 4.564764421538444
638 sigma_

In [14]:
sigma_true = res["sigma_test"]
rel = abs_rel_err

bins = [10, 20, 50, 100, 200, 500, 1000, 2000]
# сделаем устойчиво: финальный бин включит всё выше
hist = []
for b0, b1 in zip(bins[:-1], bins[1:]):
    mask = (sigma_true >= b0) & (sigma_true < b1)
    if mask.sum() > 0:
        # возьмем median и RMSE rel в этом диапазоне
        rel_rmse = np.sqrt(np.mean(rel[mask]**2))
        rel_med = np.median(rel[mask])
        hist.append((b0, b1, mask.sum(), rel_med, rel_rmse))

print("rel error by sigma_true bins: b0-b1 | count | median | rel_RMSE")
for row in hist:
    print(row)


rel error by sigma_true bins: b0-b1 | count | median | rel_RMSE
(10, 20, np.int64(1), np.float64(20.9825927734375), np.float64(20.9825927734375))
(20, 50, np.int64(19), np.float64(4.417519124348958), np.float64(4.814652401060605))
(50, 100, np.int64(58), np.float64(1.6047985402149472), np.float64(1.836229374375741))
(100, 200, np.int64(181), np.float64(0.5595284598214286), np.float64(0.6552449970756636))
(200, 500, np.int64(253), np.float64(0.2066324349119975), np.float64(0.27764324450800254))
(500, 1000, np.int64(172), np.float64(0.5203040676002375), np.float64(0.49878000601131733))
(1000, 2000, np.int64(6), np.float64(0.6565527567148662), np.float64(0.6797379659654986))


In [15]:
y_true = -np.log10(res["sigma_test"])
y_pred = -np.log10(res["sigma_pred_test"])  # обратная проверка (для сравнения в y space)

# ошибка в y-space:
err_y = y_pred - y_true
print("err_y stats:", np.median(np.abs(err_y)), np.mean(np.abs(err_y)), np.sqrt(np.mean(err_y**2)))


err_y stats: 0.20722750795029543 0.232580695613743 0.2838917824738606


In [16]:
import numpy as np

sigma_true = res["sigma_test"]
sigma_pred = res["sigma_pred_test"]

mask = sigma_true > 100

rmse_rel_gt100 = np.sqrt(np.mean(((sigma_pred[mask] - sigma_true[mask]) / sigma_true[mask])**2))
mae_gt100 = np.mean(np.abs(sigma_pred[mask] - sigma_true[mask]))
rmse_abs_gt100 = np.sqrt(np.mean((sigma_pred[mask] - sigma_true[mask])**2))

print("N points sigma>100:", mask.sum())
print("MAE (MPa) sigma>100:", mae_gt100)
print("RMSE (MPa) sigma>100:", rmse_abs_gt100)
print("RMSE_relative sigma>100:", rmse_rel_gt100)


N points sigma>100: 609
MAE (MPa) sigma>100: 155.8530973155273
RMSE (MPa) sigma>100: 212.99151052213227
RMSE_relative sigma>100: 0.4765559578545203
